# 08 Clinical Data Generalization

This notebook evaluates whether the public-data-selected optic disc/cup segmentation recipe from Notebook 07 transfers to the clinical annotation set.

The quantitative clinical subset is derived from annotated PSD composites. Red and blue clinical annotation overlays are converted into approximate optic disc/cup masks, while the overlay lines are removed from the model input by inpainting. The notebook commits only aggregate de-identified summaries; raw clinical files, derived masks, cleaned clinical images, QA sheets, and private manifests remain under ignored local data directories.


## 08.01 — Imports

Import general utilities used throughout the clinical generalization notebook.


In [1]:
# 08.01 — Imports
from __future__ import annotations

import copy
import json
import math
import os
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from torch.utils.data import DataLoader, Dataset

pd.options.display.max_columns = 140
pd.options.display.width = 200

print("Imports: OK")


Imports: OK


## 08.02 — Project root and source path setup

Find the repository root and make local source modules importable from the notebook kernel.


In [2]:
# 08.02 — Project root and source path setup
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or terminal working directory."""
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / ".git").exists() and (candidate / "src").exists():
            return candidate

    raise FileNotFoundError("Could not find project root containing .git and src/.")


PROJECT_ROOT = find_project_root()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.chdir(PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT.name}")
print("Source path:  src")


Project root: Automated-Glaucoma-Screening-Using-AI-Enhanced-Ophthalmoscopy
Source path:  src


## 08.03 — Phase-08 path setup

Define private clinical-data paths and committed aggregate-report paths.


In [3]:
# 08.03 — Phase-08 path setup
PHASE_NAME = "08_clinical_data_generalization"

CLINICAL_ROOT = PROJECT_ROOT / "data/external/clinical"
CLINICAL_PSD_ROOT = CLINICAL_ROOT / "Annotated slides"

PRIVATE_CLINICAL_DIR = PROJECT_ROOT / "data/interim/private_clinical_generalization"
PRIVATE_CLEAN_DIR = PRIVATE_CLINICAL_DIR / "clean_images"
PRIVATE_MASK_DIR = PRIVATE_CLINICAL_DIR / "derived_masks"
PRIVATE_QA_DIR = PRIVATE_CLINICAL_DIR / "qa"
PRIVATE_CLINICAL_MANIFEST_PATH = PRIVATE_CLINICAL_DIR / "clinical_psd_derived_mask_manifest_private.csv"
PRIVATE_CLINICAL_EXTRACTION_SUMMARY_PATH = PRIVATE_CLINICAL_DIR / "clinical_psd_extraction_summary_private.csv"
PRIVATE_QA_CONTACT_SHEET_PATH = PRIVATE_QA_DIR / "clinical_psd_extraction_contact_sheet_private.png"

DATA_AUDIT_DIR = PROJECT_ROOT / "reports/data_audit"
TRAINING_REPORTS_DIR = PROJECT_ROOT / "reports/training"

FINAL_SELECTION_SUMMARY_PATH = TRAINING_REPORTS_DIR / "final_model_selection_summary.csv"

CLINICAL_EXTRACTION_SUMMARY_PATH = DATA_AUDIT_DIR / "clinical_psd_annotation_extraction_summary.csv"
CLINICAL_DATASET_SUMMARY_PATH = DATA_AUDIT_DIR / "clinical_generalization_dataset_summary.csv"

CLINICAL_REBUILD_HISTORY_PATH = TRAINING_REPORTS_DIR / "clinical_generalization_model_rebuild_history.csv"
CLINICAL_REBUILD_METADATA_PATH = TRAINING_REPORTS_DIR / "clinical_generalization_model_rebuild_metadata.csv"
CLINICAL_EVALUATION_SUMMARY_PATH = TRAINING_REPORTS_DIR / "clinical_generalization_evaluation_summary.csv"
CLINICAL_PATIENT_WEIGHTED_SUMMARY_PATH = TRAINING_REPORTS_DIR / "clinical_generalization_patient_weighted_summary.csv"

for directory in [
    PRIVATE_CLEAN_DIR,
    PRIVATE_MASK_DIR,
    PRIVATE_QA_DIR,
    DATA_AUDIT_DIR,
    TRAINING_REPORTS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

path_summary = pd.DataFrame(
    [
        {"path_name": "clinical_psd_root", "path": str(CLINICAL_PSD_ROOT.relative_to(PROJECT_ROOT)), "committed": False},
        {"path_name": "private_clinical_dir", "path": str(PRIVATE_CLINICAL_DIR.relative_to(PROJECT_ROOT)), "committed": False},
        {"path_name": "clinical_extraction_summary", "path": str(CLINICAL_EXTRACTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)), "committed": True},
        {"path_name": "clinical_dataset_summary", "path": str(CLINICAL_DATASET_SUMMARY_PATH.relative_to(PROJECT_ROOT)), "committed": True},
        {"path_name": "clinical_evaluation_summary", "path": str(CLINICAL_EVALUATION_SUMMARY_PATH.relative_to(PROJECT_ROOT)), "committed": True},
    ]
)

display(path_summary)


,path_name,path,committed
0,clinical_psd_root,data/external/clinical/Annotated slides,False
1,private_clinical_dir,data/interim/private_clinical_generalization,False
2,clinical_extraction_summary,reports/data_audit/clinical_psd_annotation_ext...,True
3,clinical_dataset_summary,reports/data_audit/clinical_generalization_dat...,True
4,clinical_evaluation_summary,reports/training/clinical_generalization_evalu...,True


## 08.04 — Source module import check

Import the clinical PSD annotation utilities and the training/evaluation utilities needed later in the notebook.


In [4]:
# 08.04 — Source module import check
from glaucoma_segmentation.clinical import (
    build_clinical_psd_annotation_dataset,
    make_public_safe_extraction_summary,
    summarize_extraction_manifest,
)

from glaucoma_segmentation.evaluation.metrics import SegMetrics, vertical_cdr_from_mask
from glaucoma_segmentation.nets.losses import DiceCELoss
from glaucoma_segmentation.nets.model_factory import build_model
from glaucoma_segmentation.training.train_loop import (
    history_to_dicts,
    run_one_epoch_with_progress,
)
from glaucoma_segmentation.utils.device import describe_device, get_device
from glaucoma_segmentation.utils.seed import seed_everything

print("Clinical and training module imports: OK")


/home/gsr3qz/.conda/envs/glaucoma-capstone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Clinical and training module imports: OK


## 08.05 — Notebook 07 selected public-data recipe

Load the final public-data model selected in Notebook 07. This recipe is the clinical-generalization candidate evaluated here.


In [5]:
# 08.05 — Notebook 07 selected public-data recipe
if not FINAL_SELECTION_SUMMARY_PATH.exists():
    raise FileNotFoundError(
        f"Missing Notebook 07 final selection summary: {FINAL_SELECTION_SUMMARY_PATH}"
    )

final_selection_summary = pd.read_csv(FINAL_SELECTION_SUMMARY_PATH)

selected_rows = final_selection_summary.loc[
    final_selection_summary["selected_for_notebook_08_clinical_generalization"].astype(bool)
].copy()

if len(selected_rows) != 1:
    raise ValueError(
        "Expected exactly one selected model from Notebook 07, "
        f"found {len(selected_rows)}."
    )

selected_model_config = selected_rows.iloc[0].to_dict()

selected_model_display = pd.DataFrame(
    [
        {
            "run_name": selected_model_config["run_name"],
            "model_name": selected_model_config["model_name"],
            "encoder_name": selected_model_config["encoder_name"],
            "strategy_name": selected_model_config["strategy_name"],
            "best_epoch_public_validation": int(selected_model_config["best_epoch_by_val_mean_foreground_dice"]),
            "public_val_mean_foreground_dice": float(selected_model_config["val_mean_foreground_dice"]),
            "public_test_mean_foreground_dice": float(selected_model_config["test_mean_foreground_dice"]),
            "public_test_disc_dice": float(selected_model_config["test_disc_dice"]),
            "public_test_cup_dice": float(selected_model_config["test_cup_dice"]),
            "public_test_cdr_mae": float(selected_model_config["test_cdr_mae"]),
            "public_test_margin_meaningful": bool(selected_model_config["meaningful_test_margin_over_next"]),
        }
    ]
)

display(selected_model_display)


,run_name,model_name,encoder_name,strategy_name,best_epoch_public_validation,public_val_mean_foreground_dice,public_test_mean_foreground_dice,public_test_disc_dice,public_test_cup_dice,public_test_cdr_mae,public_test_margin_meaningful
0,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,4,0.822927,0.817976,0.83995,0.796002,0.063602,False


## 08.06 — Privacy-safe clinical PSD annotation extraction

Process the annotated clinical PSD composites into private cleaned inputs and private derived label masks. Commit only the aggregate extraction summary.


In [6]:
# 08.06 — Privacy-safe clinical PSD annotation extraction
if not CLINICAL_PSD_ROOT.exists():
    raise FileNotFoundError(
        f"Clinical PSD root not found: {CLINICAL_PSD_ROOT}. "
        "This notebook expects the private clinical PSD files to remain outside version control."
    )

clinical_extraction = build_clinical_psd_annotation_dataset(
    psd_root=CLINICAL_PSD_ROOT,
    output_dir=PRIVATE_CLINICAL_DIR,
    project_root=PROJECT_ROOT,
    qa_tile_limit=24,
)

clinical_private_manifest = clinical_extraction["manifest"]
clinical_private_summary = clinical_extraction["summary"]
clinical_public_summary = clinical_extraction["public_summary"]

clinical_public_summary.to_csv(CLINICAL_EXTRACTION_SUMMARY_PATH, index=False)

display(clinical_public_summary)

print("Private clinical extraction outputs:")
print(f"  private manifest: {PRIVATE_CLINICAL_MANIFEST_PATH.relative_to(PROJECT_ROOT)}")
print(f"  private summary:  {PRIVATE_CLINICAL_EXTRACTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"  private QA sheet: {PRIVATE_QA_CONTACT_SHEET_PATH.relative_to(PROJECT_ROOT)}")
print()
print(f"Committed aggregate summary: {CLINICAL_EXTRACTION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")


,dataset,source_type,mask_source,contains_private_paths,contains_patient_hashes,metric,value
0,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,psd_files_processed,135.000000
1,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,read_ok_count,135.000000
2,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,mask_ready_count,59.000000
3,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,mask_ready_rate,0.437037
4,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,unique_patient_count,35.000000
5,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,eye_od_count,56.000000
6,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,eye_os_count,54.000000
7,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,eye_unknown_count,25.000000
8,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,mask_ready_unique_patient_count,20.000000
9,clinical_psd_annotations,annotated_psd_composites,red_blue_overlay_extraction,False,False,mask_ready_eye_od_count,21.000000


Private clinical extraction outputs:
  private manifest: data/interim/private_clinical_generalization/clinical_psd_derived_mask_manifest_private.csv
  private summary:  data/interim/private_clinical_generalization/clinical_psd_extraction_summary_private.csv
  private QA sheet: data/interim/private_clinical_generalization/qa/clinical_psd_extraction_contact_sheet_private.png

Committed aggregate summary: reports/data_audit/clinical_psd_annotation_extraction_summary.csv


## 08.07 — Clinical PSD extraction readiness summary

Summarize mask-readiness without exposing raw filenames, raw paths, or patient-level identifiers.


In [7]:
# 08.07 — Clinical PSD extraction readiness summary
if clinical_private_manifest.empty:
    raise ValueError("No clinical PSD files were processed.")

mask_ready_count = int(clinical_private_manifest["mask_ready"].sum())

if mask_ready_count == 0:
    raise ValueError(
        "No clinical PSD files were mask-ready after annotation extraction. "
        "Review the private QA sheet before continuing."
    )

status_counts = (
    clinical_private_manifest
    .groupby(["extraction_status", "mask_ready"], dropna=False)
    .size()
    .rename("file_count")
    .reset_index()
    .sort_values(["mask_ready", "file_count"], ascending=[False, False])
)

eye_counts = (
    clinical_private_manifest
    .groupby(["eye", "mask_ready"], dropna=False)
    .size()
    .rename("file_count")
    .reset_index()
    .sort_values(["mask_ready", "eye"], ascending=[False, True])
)

ready_manifest = clinical_private_manifest.loc[clinical_private_manifest["mask_ready"]].copy()

readiness_summary = pd.DataFrame(
    [
        {"item": "processed_psd_files", "value": int(len(clinical_private_manifest))},
        {"item": "mask_ready_psd_files", "value": int(mask_ready_count)},
        {"item": "mask_ready_rate", "value": float(mask_ready_count / len(clinical_private_manifest))},
        {"item": "unique_patient_groups_all", "value": int(clinical_private_manifest["patient_hash"].nunique())},
        {"item": "unique_patient_groups_mask_ready", "value": int(ready_manifest["patient_hash"].nunique())},
        {"item": "private_qa_sheet_exists", "value": PRIVATE_QA_CONTACT_SHEET_PATH.exists()},
    ]
)

display(readiness_summary)
display(status_counts)
display(eye_counts)

print("Clinical PSD extraction is ready for quantitative evaluation.")
print("Review the private QA contact sheet before interpreting metrics:")
print(f"  {PRIVATE_QA_CONTACT_SHEET_PATH.relative_to(PROJECT_ROOT)}")


,item,value
0,processed_psd_files,135
1,mask_ready_psd_files,59
2,mask_ready_rate,0.437037
3,unique_patient_groups_all,35
4,unique_patient_groups_mask_ready,20
5,private_qa_sheet_exists,True


,extraction_status,mask_ready,file_count
4,ok,True,59
0,fallback_nearest_blue,False,45
3,ok,False,26
1,no_blue_component,False,4
2,no_red_component,False,1


,eye,mask_ready,file_count
1,OD,True,21
3,OS,True,13
4,unknown,True,25
0,OD,False,35
2,OS,False,41


Clinical PSD extraction is ready for quantitative evaluation.
Review the private QA contact sheet before interpreting metrics:
  data/interim/private_clinical_generalization/qa/clinical_psd_extraction_contact_sheet_private.png


## 08.08 — Clinical generalization dataset summary

Create a committed aggregate dataset summary for the mask-ready PSD-derived clinical subset.


In [8]:
# 08.08 — Clinical generalization dataset summary
ready_manifest = clinical_private_manifest.loc[clinical_private_manifest["mask_ready"]].copy()

clinical_dataset_summary = pd.DataFrame(
    [
        {
            "dataset_label": "clinical_psd_derived_mask_ready",
            "source": "annotated_clinical_psd_composites",
            "used_for": "quantitative_clinical_generalization",
            "mask_source": "red_blue_annotation_overlay_extraction",
            "model_input_source": "annotation_inpainted_clinical_psd_composite",
            "rows": int(len(ready_manifest)),
            "unique_patient_groups": int(ready_manifest["patient_hash"].nunique()),
            "eye_od_rows": int((ready_manifest["eye"] == "OD").sum()),
            "eye_os_rows": int((ready_manifest["eye"] == "OS").sum()),
            "eye_unknown_rows": int((ready_manifest["eye"] == "unknown").sum()),
            "contains_raw_clinical_paths": False,
            "contains_patient_hashes": False,
            "private_files_committed": False,
            "image_size_for_model": "(256, 256)",
        }
    ]
)

clinical_dataset_summary.to_csv(CLINICAL_DATASET_SUMMARY_PATH, index=False)

display(clinical_dataset_summary)
print(f"Saved dataset summary: {CLINICAL_DATASET_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")


,dataset_label,source,used_for,mask_source,model_input_source,rows,unique_patient_groups,eye_od_rows,eye_os_rows,eye_unknown_rows,contains_raw_clinical_paths,contains_patient_hashes,private_files_committed,image_size_for_model
0,clinical_psd_derived_mask_ready,annotated_clinical_psd_composites,quantitative_clinical_generalization,red_blue_annotation_overlay_extraction,annotation_inpainted_clinical_psd_composite,59,20,21,13,25,False,False,False,"(256, 256)"


Saved dataset summary: reports/data_audit/clinical_generalization_dataset_summary.csv


## 08.09 — Notebook 08 setup handoff summary

Confirm that the clinical PSD-derived quantitative subset is ready. The next section rebuilds the selected public-data model recipe and evaluates clinical generalization.


In [9]:
# 08.09 — Notebook 08 setup handoff summary
notebook_08_setup_handoff = pd.DataFrame(
    [
        {
            "item": "selected_public_model",
            "status": "ready",
            "detail": (
                f"{selected_model_config['run_name']} "
                f"({selected_model_config['model_name']} / {selected_model_config['encoder_name']}; "
                f"strategy={selected_model_config['strategy_name']})"
            ),
        },
        {
            "item": "clinical_annotation_source",
            "status": "ready",
            "detail": "Annotated clinical PSD composites processed into private cleaned images and private derived masks.",
        },
        {
            "item": "mask_ready_clinical_subset",
            "status": "ready",
            "detail": (
                f"{len(ready_manifest)} mask-ready PSD-derived samples across "
                f"{ready_manifest['patient_hash'].nunique()} patient/encounter groups."
            ),
        },
        {
            "item": "privacy_controls",
            "status": "ready",
            "detail": "Raw clinical files, private manifests, cleaned images, derived masks, and QA sheets remain under ignored local paths.",
        },
        {
            "item": "next_step",
            "status": "ready",
            "detail": "Rebuild the selected public-data model recipe and evaluate it on the PSD-derived clinical subset.",
        },
    ]
)

display(notebook_08_setup_handoff)


,item,status,detail
0,selected_public_model,ready,final_unetplusplus_resnet18_small_affine_virtu...
1,clinical_annotation_source,ready,Annotated clinical PSD composites processed in...
2,mask_ready_clinical_subset,ready,59 mask-ready PSD-derived samples across 20 pa...
3,privacy_controls,ready,"Raw clinical files, private manifests, cleaned..."
4,next_step,ready,Rebuild the selected public-data model recipe ...


## 08.10 — Clinical evaluation imports and rebuild configuration

Import the source-backed dataset/evaluation utilities and configure the selected Notebook 07 public-data recipe for clinical generalization.


In [10]:
# 08.10 — Clinical evaluation imports and rebuild configuration
from glaucoma_segmentation.augmentation import (
    build_virtual_synthetic_expansion_dataset,
    summarize_virtual_synthetic_expansion,
)
from glaucoma_segmentation.clinical import (
    ClinicalDerivedMaskDataset,
    evaluate_model_on_clinical_loader,
    summarize_image_level_clinical_metrics,
    summarize_patient_weighted_clinical_metrics,
)
from glaucoma_segmentation.data.dataloaders import make_segmentation_datasets
from glaucoma_segmentation.training.train_loop import model_forward

SEED = 42
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8

try:
    AVAILABLE_CPU_CORES = len(os.sched_getaffinity(0))
except AttributeError:
    AVAILABLE_CPU_CORES = os.cpu_count() or 1

NUM_WORKERS = min(4, max(1, AVAILABLE_CPU_CORES))
PIN_MEMORY = bool(torch.cuda.is_available())

EPOCHS = 5
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MANIFEST_PATH = PROJECT_ROOT / "data/processed/manifests/combined_segmentation_manifest_with_splits.csv"

PRIVATE_CLINICAL_IMAGE_LEVEL_METRICS_PATH = (
    PRIVATE_CLINICAL_DIR / "clinical_generalization_image_level_metrics_private.csv"
)
PRIVATE_MODEL_REBUILD_CHECKPOINT_PATH = (
    PRIVATE_CLINICAL_DIR / "selected_public_model_rebuild_best_state_private.pt"
)

OVERWRITE_MODEL_REBUILD = False

def value_or_default(mapping: dict[str, Any], key: str, default: Any) -> Any:
    value = mapping.get(key, default)
    if value is None:
        return default
    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass
    return value


def normalize_encoder_weights(value: Any) -> Any:
    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except TypeError:
        pass

    value_string = str(value).strip()
    if value_string.lower() in {"", "none", "nan", "null"}:
        return None

    return value_string


SELECTED_RUN_NAME = str(selected_model_config["run_name"])
SELECTED_MODEL_NAME = str(selected_model_config["model_name"])
SELECTED_ENCODER_NAME = str(selected_model_config["encoder_name"])
SELECTED_ENCODER_WEIGHTS = normalize_encoder_weights(
    value_or_default(selected_model_config, "encoder_weights", None)
)
SELECTED_STRATEGY_NAME = str(selected_model_config["strategy_name"])
SELECTED_SYNTHETIC_COPY_COUNT = int(
    float(value_or_default(selected_model_config, "synthetic_copy_count", 1))
)
NOTEBOOK_07_BEST_EPOCH = int(
    float(selected_model_config["best_epoch_by_val_mean_foreground_dice"])
)

seed_everything(SEED)

DEVICE = get_device()
try:
    DEVICE_INFO = describe_device(DEVICE)
except TypeError:
    DEVICE_INFO = describe_device()

clinical_eval_config = pd.DataFrame(
    [
        {"setting": "selected_run_name", "value": SELECTED_RUN_NAME},
        {"setting": "model_name", "value": SELECTED_MODEL_NAME},
        {"setting": "encoder_name", "value": SELECTED_ENCODER_NAME},
        {"setting": "encoder_weights", "value": "none" if SELECTED_ENCODER_WEIGHTS is None else SELECTED_ENCODER_WEIGHTS},
        {"setting": "strategy_name", "value": SELECTED_STRATEGY_NAME},
        {"setting": "synthetic_copy_count", "value": SELECTED_SYNTHETIC_COPY_COUNT},
        {"setting": "notebook_07_best_epoch", "value": NOTEBOOK_07_BEST_EPOCH},
        {"setting": "rebuild_epochs", "value": EPOCHS},
        {"setting": "image_size", "value": IMAGE_SIZE},
        {"setting": "batch_size", "value": BATCH_SIZE},
        {"setting": "num_workers", "value": NUM_WORKERS},
        {"setting": "pin_memory", "value": PIN_MEMORY},
        {"setting": "device", "value": str(DEVICE)},
        {"setting": "device_info", "value": str(DEVICE_INFO)},
    ]
)

display(clinical_eval_config)

print("Clinical evaluation configuration: OK")


,setting,value
0,selected_run_name,final_unetplusplus_resnet18_small_affine_virtu...
1,model_name,unetplusplus
2,encoder_name,resnet18
3,encoder_weights,none
4,strategy_name,small_affine
5,synthetic_copy_count,1
6,notebook_07_best_epoch,4
7,rebuild_epochs,5
8,image_size,"(256, 256)"
9,batch_size,8


Clinical evaluation configuration: OK


## 08.11 — Build public rebuild datasets and clinical evaluation dataset

Recreate the public-data training/validation pipeline for the selected Notebook 07 recipe and build the PSD-derived clinical evaluation loader.


In [11]:
# 08.11 — Build public rebuild datasets and clinical evaluation dataset
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Missing public split manifest: {MANIFEST_PATH}")

if "clinical_private_manifest" not in globals():
    if not PRIVATE_CLINICAL_MANIFEST_PATH.exists():
        raise FileNotFoundError(
            "Clinical private manifest is missing. Run 08.06 before this cell."
        )
    clinical_private_manifest = pd.read_csv(PRIVATE_CLINICAL_MANIFEST_PATH)

ready_manifest = clinical_private_manifest.loc[
    clinical_private_manifest["mask_ready"].astype(bool)
].copy()

if ready_manifest.empty:
    raise ValueError("No mask-ready clinical samples available for evaluation.")

public_datasets = make_segmentation_datasets(
    manifest_path=MANIFEST_PATH,
    splits=("train", "val", "test"),
    image_size=IMAGE_SIZE,
    validate_paths=True,
    validate_masks=True,
)

selected_train_dataset = build_virtual_synthetic_expansion_dataset(
    base_dataset=public_datasets.train,
    strategy_names=SELECTED_STRATEGY_NAME,
    copy_count=SELECTED_SYNTHETIC_COPY_COUNT,
    base_seed=SEED,
    include_original=True,
    add_metadata=True,
)

selected_virtual_summary = summarize_virtual_synthetic_expansion(selected_train_dataset)

clinical_dataset = ClinicalDerivedMaskDataset(
    ready_manifest,
    project_root=PROJECT_ROOT,
    image_size=IMAGE_SIZE,
    require_mask_ready=True,
    validate_paths=True,
)

def build_loader(
    dataset: Any,
    *,
    batch_size: int,
    shuffle: bool,
    num_workers: int,
    pin_memory: bool,
    generator: torch.Generator | None = None,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        generator=generator,
        drop_last=False,
        persistent_workers=num_workers > 0,
    )


train_generator = torch.Generator()
train_generator.manual_seed(SEED + 808)

selected_train_loader = build_loader(
    selected_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    generator=train_generator,
)

public_val_loader = build_loader(
    public_datasets.val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

clinical_loader = build_loader(
    clinical_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

clinical_batch = next(iter(clinical_loader))

dataset_rebuild_summary = pd.DataFrame(
    [
        {
            "dataset_label": "public_train_virtual_synthetic_rebuild",
            "rows": len(selected_train_dataset),
            "patient_groups": np.nan,
            "used_for": "model_rebuild_training",
            "source": "public_fundus_train_split",
            "mask_source": "public_ground_truth",
            "contains_private_paths": False,
            "contains_patient_hashes": False,
        },
        {
            "dataset_label": "public_validation_original_rebuild",
            "rows": len(public_datasets.val),
            "patient_groups": np.nan,
            "used_for": "model_rebuild_validation",
            "source": "public_fundus_validation_split",
            "mask_source": "public_ground_truth",
            "contains_private_paths": False,
            "contains_patient_hashes": False,
        },
        {
            "dataset_label": "clinical_psd_derived_mask_ready",
            "rows": len(clinical_dataset),
            "patient_groups": int(ready_manifest["patient_hash"].nunique()),
            "used_for": "clinical_generalization_evaluation",
            "source": "clinical_annotated_psd_composites",
            "mask_source": "red_blue_overlay_extraction",
            "contains_private_paths": False,
            "contains_patient_hashes": False,
        },
    ]
)

clinical_loader_smoke = pd.DataFrame(
    [
        {
            "loader": "clinical_psd_derived_mask_ready",
            "batch_image_shape": tuple(clinical_batch["image"].shape),
            "batch_mask_shape": tuple(clinical_batch["mask"].shape),
            "image_dtype": str(clinical_batch["image"].dtype),
            "mask_dtype": str(clinical_batch["mask"].dtype),
            "image_min": float(clinical_batch["image"].min()),
            "image_max": float(clinical_batch["image"].max()),
            "mask_values": sorted(int(value) for value in torch.unique(clinical_batch["mask"]).tolist()),
            "sample_hash_present": "sample_hash" in clinical_batch,
            "patient_hash_present": "patient_hash" in clinical_batch,
        }
    ]
)

display(dataset_rebuild_summary)
display(pd.DataFrame([selected_virtual_summary]))
display(clinical_loader_smoke)

print("Public rebuild datasets and clinical evaluation dataset: OK")


,dataset_label,rows,patient_groups,used_for,source,mask_source,contains_private_paths,contains_patient_hashes
0,public_train_virtual_synthetic_rebuild,3822,NaN,model_rebuild_training,public_fundus_train_split,public_ground_truth,False,False
1,public_validation_original_rebuild,725,NaN,model_rebuild_validation,public_fundus_validation_split,public_ground_truth,False,False
2,clinical_psd_derived_mask_ready,59,20.0,clinical_generalization_evaluation,clinical_annotated_psd_composites,red_blue_overlay_extraction,False,False


,base_rows,original_rows_exposed,synthetic_rows_exposed,total_rows_exposed,base_seed,include_original,strategies,copy_counts
0,1911,1911,1911,3822,42,True,small_affine,1


,loader,batch_image_shape,batch_mask_shape,image_dtype,mask_dtype,image_min,image_max,mask_values,sample_hash_present,patient_hash_present
0,clinical_psd_derived_mask_ready,"(8, 3, 256, 256)","(8, 256, 256)",torch.float32,torch.int64,0.0,1.0,"[0, 1, 2]",True,True


Public rebuild datasets and clinical evaluation dataset: OK


## 08.12 — Rebuild the selected public-data model recipe

Notebook 07 saved the selected model configuration and evaluation summaries, but the committed repository does not include reusable trained model weights. Therefore, this cell reconstructs the selected Notebook 07 public-data recipe, trains it only on the public training split with virtual synthetic add-back, restores the best public-validation epoch, and saves the resulting checkpoint under an ignored private path.

No clinical samples are used for training, validation, tuning, or model selection in this notebook. The clinical PSD-derived subset is used only after this rebuild step for generalization evaluation.


In [12]:
# 08.12 — Rebuild the selected public-data model
def as_float(value: Any) -> float:
    if torch.is_tensor(value):
        return float(value.detach().cpu().item())
    return float(value)


def mean_foreground_dice_from_metrics(metrics: dict[str, Any]) -> float:
    return float((as_float(metrics["disc_dice"]) + as_float(metrics["cup_dice"])) / 2.0)


def build_selected_model() -> torch.nn.Module:
    return build_model(
        model_name=SELECTED_MODEL_NAME,
        in_channels=3,
        classes=3,
        encoder_name=SELECTED_ENCODER_NAME,
        encoder_weights=SELECTED_ENCODER_WEIGHTS,
        activation=None,
    )


def run_rebuild_epoch(
    model: torch.nn.Module,
    loader: DataLoader,
    *,
    criterion: torch.nn.Module,
    optimizer: torch.optim.Optimizer | None,
    device: torch.device,
    phase: str,
    epoch: int,
) -> dict[str, Any]:
    is_train = phase == "train"
    model.train(is_train)

    metrics = SegMetrics()
    running_loss = 0.0
    n_batches = 0
    n_images = 0

    for batch in loader:
        images = batch["image"].to(device, non_blocking=True).float()
        masks = batch["mask"].to(device, non_blocking=True).long()

        if is_train:
            if optimizer is None:
                raise ValueError("Optimizer is required for training phase.")
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(is_train):
            logits = model_forward(model, images)
            loss = criterion(logits, masks)

            if is_train:
                loss.backward()
                optimizer.step()

        batch_size = int(images.shape[0])
        running_loss += float(loss.detach().cpu()) * batch_size
        n_batches += 1
        n_images += batch_size
        metrics.update(logits.detach(), masks.detach())

    if n_images == 0:
        raise ValueError(f"No images were processed during {phase} epoch {epoch}.")

    metric_summary = metrics.compute()
    disc_dice = as_float(metric_summary["disc_dice"])
    cup_dice = as_float(metric_summary["cup_dice"])
    cdr_mae = as_float(metric_summary["cdr_mae"])
    mean_foreground_dice = float((disc_dice + cup_dice) / 2.0)

    return {
        "run_name": SELECTED_RUN_NAME,
        "model_name": SELECTED_MODEL_NAME,
        "encoder_name": SELECTED_ENCODER_NAME,
        "strategy_name": SELECTED_STRATEGY_NAME,
        "phase": phase,
        "epoch": int(epoch),
        "loss": float(running_loss / n_images),
        "disc_dice": disc_dice,
        "cup_dice": cup_dice,
        "mean_foreground_dice": mean_foreground_dice,
        "cdr_mae": cdr_mae,
        "n_images": int(n_images),
        "n_batches": int(n_batches),
    }


rebuild_start = time.time()

selected_model = build_selected_model().to(DEVICE)
criterion = DiceCELoss()
optimizer = torch.optim.AdamW(
    selected_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

history_rows: list[dict[str, Any]] = []
best_state: dict[str, torch.Tensor] | None = None
best_val_mean_foreground_dice = -float("inf")
best_epoch = None

checkpoint_can_be_loaded = (
    PRIVATE_MODEL_REBUILD_CHECKPOINT_PATH.exists()
    and CLINICAL_REBUILD_HISTORY_PATH.exists()
    and CLINICAL_REBUILD_METADATA_PATH.exists()
    and not OVERWRITE_MODEL_REBUILD
)

if checkpoint_can_be_loaded:
    selected_model.load_state_dict(
        torch.load(PRIVATE_MODEL_REBUILD_CHECKPOINT_PATH, map_location=DEVICE)
    )
    rebuild_history = pd.read_csv(CLINICAL_REBUILD_HISTORY_PATH)
    rebuild_metadata = pd.read_csv(CLINICAL_REBUILD_METADATA_PATH)
    print("Loaded existing private model-rebuild checkpoint and committed rebuild summaries.")
else:
    for epoch in range(1, EPOCHS + 1):
        train_row = run_rebuild_epoch(
            selected_model,
            selected_train_loader,
            criterion=criterion,
            optimizer=optimizer,
            device=DEVICE,
            phase="train",
            epoch=epoch,
        )

        val_row = run_rebuild_epoch(
            selected_model,
            public_val_loader,
            criterion=criterion,
            optimizer=None,
            device=DEVICE,
            phase="val",
            epoch=epoch,
        )

        history_rows.extend([train_row, val_row])

        print(
            f"epoch={epoch:02d} "
            f"train_loss={train_row['loss']:.4f} "
            f"val_mean_fg_dice={val_row['mean_foreground_dice']:.4f} "
            f"val_disc={val_row['disc_dice']:.4f} "
            f"val_cup={val_row['cup_dice']:.4f} "
            f"val_cdr_mae={val_row['cdr_mae']:.4f}"
        )

        if val_row["mean_foreground_dice"] > best_val_mean_foreground_dice:
            best_val_mean_foreground_dice = float(val_row["mean_foreground_dice"])
            best_epoch = int(epoch)
            best_state = copy.deepcopy(selected_model.state_dict())

    if best_state is None or best_epoch is None:
        raise RuntimeError("Model rebuild did not produce a best validation state.")

    selected_model.load_state_dict(best_state)
    torch.save(best_state, PRIVATE_MODEL_REBUILD_CHECKPOINT_PATH)

    rebuild_history = pd.DataFrame(history_rows)
    rebuild_history.to_csv(CLINICAL_REBUILD_HISTORY_PATH, index=False)

    best_val_row = (
        rebuild_history
        .loc[rebuild_history["phase"] == "val"]
        .sort_values(["mean_foreground_dice", "cup_dice", "disc_dice"], ascending=[False, False, False])
        .head(1)
        .iloc[0]
        .to_dict()
    )

    rebuild_minutes = (time.time() - rebuild_start) / 60.0

    rebuild_metadata = pd.DataFrame(
        [
            {
                "run_name": SELECTED_RUN_NAME,
                "model_name": SELECTED_MODEL_NAME,
                "encoder_name": SELECTED_ENCODER_NAME,
                "encoder_weights": "none" if SELECTED_ENCODER_WEIGHTS is None else SELECTED_ENCODER_WEIGHTS,
                "strategy_name": SELECTED_STRATEGY_NAME,
                "synthetic_copy_count": SELECTED_SYNTHETIC_COPY_COUNT,
                "epochs_requested": EPOCHS,
                "best_epoch_by_public_val_mean_foreground_dice": int(best_val_row["epoch"]),
                "best_public_val_mean_foreground_dice": float(best_val_row["mean_foreground_dice"]),
                "best_public_val_disc_dice": float(best_val_row["disc_dice"]),
                "best_public_val_cup_dice": float(best_val_row["cup_dice"]),
                "best_public_val_cdr_mae": float(best_val_row["cdr_mae"]),
                "public_train_original_rows": int(len(public_datasets.train)),
                "public_train_virtual_rows": int(len(selected_train_dataset)),
                "public_validation_rows": int(len(public_datasets.val)),
                "clinical_evaluation_rows": int(len(clinical_dataset)),
                "clinical_evaluation_patient_groups": int(ready_manifest["patient_hash"].nunique()),
                "private_checkpoint_saved": bool(PRIVATE_MODEL_REBUILD_CHECKPOINT_PATH.exists()),
                "model_rebuild_minutes": float(rebuild_minutes),
                "device": str(DEVICE),
            }
        ]
    )

    rebuild_metadata.to_csv(CLINICAL_REBUILD_METADATA_PATH, index=False)

display(rebuild_history)
display(rebuild_metadata)

print(f"Saved rebuild history:  {CLINICAL_REBUILD_HISTORY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved rebuild metadata: {CLINICAL_REBUILD_METADATA_PATH.relative_to(PROJECT_ROOT)}")
print("Selected public-data model rebuild: OK")


epoch=01 train_loss=1.0192 val_mean_fg_dice=0.7407 val_disc=0.7675 val_cup=0.7140 val_cdr_mae=0.1068
epoch=02 train_loss=0.3722 val_mean_fg_dice=0.7445 val_disc=0.7457 val_cup=0.7433 val_cdr_mae=0.1067
epoch=03 train_loss=0.1814 val_mean_fg_dice=0.8193 val_disc=0.8402 val_cup=0.7984 val_cdr_mae=0.0681
epoch=04 train_loss=0.1426 val_mean_fg_dice=0.7978 val_disc=0.8090 val_cup=0.7867 val_cdr_mae=0.0818
epoch=05 train_loss=0.1262 val_mean_fg_dice=0.8300 val_disc=0.8519 val_cup=0.8081 val_cdr_mae=0.0651


,run_name,model_name,encoder_name,strategy_name,phase,epoch,loss,disc_dice,cup_dice,mean_foreground_dice,cdr_mae,n_images,n_batches
0,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,train,1,1.019207,0.593927,0.419930,0.506929,0.439799,3822,478
1,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,val,1,0.610685,0.767491,0.713956,0.740723,0.106849,725,91
2,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,train,2,0.372179,0.786343,0.689664,0.738003,0.110500,3822,478
3,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,val,2,0.254547,0.745722,0.743279,0.744500,0.106717,725,91
4,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,train,3,0.181414,0.830124,0.732986,0.781555,0.090174,3822,478
5,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,val,3,0.155745,0.840197,0.798407,0.819302,0.068092,725,91
6,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,train,4,0.142586,0.852588,0.767080,0.809834,0.079072,3822,478
7,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,val,4,0.156655,0.809013,0.786679,0.797846,0.081835,725,91
8,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,train,5,0.126195,0.863552,0.780465,0.822008,0.073854,3822,478
9,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,val,5,0.129633,0.851884,0.808141,0.830012,0.065088,725,91


,run_name,model_name,encoder_name,encoder_weights,strategy_name,synthetic_copy_count,epochs_requested,best_epoch_by_public_val_mean_foreground_dice,best_public_val_mean_foreground_dice,best_public_val_disc_dice,best_public_val_cup_dice,best_public_val_cdr_mae,public_train_original_rows,public_train_virtual_rows,public_validation_rows,clinical_evaluation_rows,clinical_evaluation_patient_groups,private_checkpoint_saved,model_rebuild_minutes,device
0,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,none,small_affine,1,5,5,0.830012,0.851884,0.808141,0.065088,1911,3822,725,59,20,True,14.68168,cuda


Saved rebuild history:  reports/training/clinical_generalization_model_rebuild_history.csv
Saved rebuild metadata: reports/training/clinical_generalization_model_rebuild_metadata.csv
Selected public-data model rebuild: OK


## 08.13 — Evaluate clinical generalization

Evaluate the rebuilt public-data model on the PSD-derived clinical subset. Image-level rows with sample and patient hashes are saved only under ignored private paths; committed outputs are aggregate-only.


In [13]:
# 08.13 — Evaluate clinical generalization
clinical_image_metrics = evaluate_model_on_clinical_loader(
    selected_model,
    clinical_loader,
    device=DEVICE,
    model_forward_fn=model_forward,
)

if clinical_image_metrics.empty:
    raise ValueError("Clinical evaluation produced zero image-level metric rows.")

clinical_image_metrics.to_csv(PRIVATE_CLINICAL_IMAGE_LEVEL_METRICS_PATH, index=False)

clinical_evaluation_summary = summarize_image_level_clinical_metrics(clinical_image_metrics)
clinical_patient_weighted_summary = summarize_patient_weighted_clinical_metrics(clinical_image_metrics)

def add_model_context(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame.insert(0, "run_name", SELECTED_RUN_NAME)
    frame.insert(1, "model_name", SELECTED_MODEL_NAME)
    frame.insert(2, "encoder_name", SELECTED_ENCODER_NAME)
    frame.insert(3, "strategy_name", SELECTED_STRATEGY_NAME)
    frame.insert(4, "clinical_mask_source", "red_blue_overlay_extraction")
    frame.insert(5, "clinical_model_input_source", "annotation_inpainted_psd_composite")
    frame.insert(6, "public_test_mean_foreground_dice", float(selected_model_config["test_mean_foreground_dice"]))
    frame.insert(7, "public_test_disc_dice", float(selected_model_config["test_disc_dice"]))
    frame.insert(8, "public_test_cup_dice", float(selected_model_config["test_cup_dice"]))
    frame.insert(9, "public_test_cdr_mae", float(selected_model_config["test_cdr_mae"]))
    frame.insert(10, "contains_private_paths", False)
    frame.insert(11, "contains_patient_hashes", False)
    return frame


clinical_evaluation_summary = add_model_context(clinical_evaluation_summary)
clinical_patient_weighted_summary = add_model_context(clinical_patient_weighted_summary)

clinical_evaluation_summary.to_csv(CLINICAL_EVALUATION_SUMMARY_PATH, index=False)
clinical_patient_weighted_summary.to_csv(CLINICAL_PATIENT_WEIGHTED_SUMMARY_PATH, index=False)

display(clinical_evaluation_summary)
display(clinical_patient_weighted_summary)

print(f"Private image-level metrics: {PRIVATE_CLINICAL_IMAGE_LEVEL_METRICS_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved image-weighted clinical summary: {CLINICAL_EVALUATION_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print(f"Saved patient-weighted clinical summary: {CLINICAL_PATIENT_WEIGHTED_SUMMARY_PATH.relative_to(PROJECT_ROOT)}")
print("Clinical generalization evaluation: OK")


,run_name,model_name,encoder_name,strategy_name,clinical_mask_source,clinical_model_input_source,public_test_mean_foreground_dice,public_test_disc_dice,public_test_cup_dice,public_test_cdr_mae,contains_private_paths,contains_patient_hashes,clinical_rows,clinical_patient_groups,clinical_eye_od_rows,clinical_eye_os_rows,clinical_eye_unknown_rows,image_weighted_disc_dice_mean,image_weighted_disc_dice_median,image_weighted_disc_dice_std,image_weighted_disc_dice_min,image_weighted_disc_dice_max,image_weighted_cup_dice_mean,image_weighted_cup_dice_median,image_weighted_cup_dice_std,image_weighted_cup_dice_min,image_weighted_cup_dice_max,image_weighted_mean_foreground_dice_mean,image_weighted_mean_foreground_dice_median,image_weighted_mean_foreground_dice_std,image_weighted_mean_foreground_dice_min,image_weighted_mean_foreground_dice_max,image_weighted_cdr_abs_error_mean,image_weighted_cdr_abs_error_median,image_weighted_cdr_abs_error_std,image_weighted_cdr_abs_error_min,image_weighted_cdr_abs_error_max
0,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,red_blue_overlay_extraction,annotation_inpainted_psd_composite,0.817976,0.83995,0.796002,0.063602,False,False,59,20,21,13,25,0.377686,0.329771,0.33458,0.0,0.949057,0.25774,0.244804,0.252447,0.0,0.768311,0.317713,0.316262,0.279574,0.0,0.858684,0.290072,0.24018,0.247957,0.002353,0.981481


,run_name,model_name,encoder_name,strategy_name,clinical_mask_source,clinical_model_input_source,public_test_mean_foreground_dice,public_test_disc_dice,public_test_cup_dice,public_test_cdr_mae,contains_private_paths,contains_patient_hashes,clinical_patient_groups,patient_files_mean,patient_files_median,patient_files_min,patient_files_max,patient_weighted_disc_dice_mean,patient_weighted_disc_dice_median,patient_weighted_disc_dice_std,patient_weighted_disc_dice_min,patient_weighted_disc_dice_max,patient_weighted_cup_dice_mean,patient_weighted_cup_dice_median,patient_weighted_cup_dice_std,patient_weighted_cup_dice_min,patient_weighted_cup_dice_max,patient_weighted_mean_foreground_dice_mean,patient_weighted_mean_foreground_dice_median,patient_weighted_mean_foreground_dice_std,patient_weighted_mean_foreground_dice_min,patient_weighted_mean_foreground_dice_max,patient_weighted_cdr_abs_error_mean,patient_weighted_cdr_abs_error_median,patient_weighted_cdr_abs_error_std,patient_weighted_cdr_abs_error_min,patient_weighted_cdr_abs_error_max
0,final_unetplusplus_resnet18_small_affine_virtu...,unetplusplus,resnet18,small_affine,red_blue_overlay_extraction,annotation_inpainted_psd_composite,0.817976,0.83995,0.796002,0.063602,False,False,20,2.95,2.0,1,10,0.267402,0.227335,0.26169,0.0,0.699215,0.216632,0.219325,0.209446,0.0,0.584507,0.242017,0.249336,0.224755,0.0,0.627657,0.283353,0.202703,0.235313,0.002353,0.790127


Private image-level metrics: data/interim/private_clinical_generalization/clinical_generalization_image_level_metrics_private.csv
Saved image-weighted clinical summary: reports/training/clinical_generalization_evaluation_summary.csv
Saved patient-weighted clinical summary: reports/training/clinical_generalization_patient_weighted_summary.csv
Clinical generalization evaluation: OK


## 08.14 — Public-to-clinical performance comparison

Compare the selected public-data test performance with the PSD-derived clinical generalization metrics. Clinical results should be interpreted as approximate because the evaluation masks are derived from annotation overlays rather than independent raw segmentation masks.


In [14]:
# 08.14 — Public-to-clinical performance comparison
image_row = clinical_evaluation_summary.iloc[0].to_dict()
patient_row = clinical_patient_weighted_summary.iloc[0].to_dict()

public_to_clinical_comparison = pd.DataFrame(
    [
        {
            "evaluation_set": "public_held_out_test",
            "weighting": "image_weighted",
            "rows_or_groups": int(selected_model_config.get("test_rows", 722)) if "test_rows" in selected_model_config else 722,
            "mean_foreground_dice": float(selected_model_config["test_mean_foreground_dice"]),
            "disc_dice": float(selected_model_config["test_disc_dice"]),
            "cup_dice": float(selected_model_config["test_cup_dice"]),
            "cdr_mae": float(selected_model_config["test_cdr_mae"]),
            "mask_source": "public_ground_truth",
        },
        {
            "evaluation_set": "clinical_psd_derived",
            "weighting": "image_weighted",
            "rows_or_groups": int(image_row["clinical_rows"]),
            "mean_foreground_dice": float(image_row["image_weighted_mean_foreground_dice_mean"]),
            "disc_dice": float(image_row["image_weighted_disc_dice_mean"]),
            "cup_dice": float(image_row["image_weighted_cup_dice_mean"]),
            "cdr_mae": float(image_row["image_weighted_cdr_abs_error_mean"]),
            "mask_source": "red_blue_overlay_extraction",
        },
        {
            "evaluation_set": "clinical_psd_derived",
            "weighting": "patient_weighted",
            "rows_or_groups": int(patient_row["clinical_patient_groups"]),
            "mean_foreground_dice": float(patient_row["patient_weighted_mean_foreground_dice_mean"]),
            "disc_dice": float(patient_row["patient_weighted_disc_dice_mean"]),
            "cup_dice": float(patient_row["patient_weighted_cup_dice_mean"]),
            "cdr_mae": float(patient_row["patient_weighted_cdr_abs_error_mean"]),
            "mask_source": "red_blue_overlay_extraction",
        },
    ]
)

display(public_to_clinical_comparison)

interpretation_rows = [
    {
        "item": "selected_public_model",
        "interpretation": (
            f"{SELECTED_RUN_NAME} was selected in Notebook 07 and rebuilt here "
            "without clinical fine-tuning."
        ),
    },
    {
        "item": "clinical_quantitative_subset",
        "interpretation": (
            f"{int(image_row['clinical_rows'])} mask-ready PSD-derived clinical samples "
            f"across {int(patient_row['clinical_patient_groups'])} patient/encounter groups were evaluated."
        ),
    },
    {
        "item": "metric_caveat",
        "interpretation": (
            "Clinical masks are approximate labels extracted from red/blue annotation overlays; "
            "therefore clinical metrics are generalization indicators rather than pristine ground-truth estimates."
        ),
    },
    {
        "item": "privacy_caveat",
        "interpretation": (
            "Committed outputs are aggregate-only. Raw clinical files, derived masks, cleaned images, "
            "private checkpoints, and image-level metrics remain under ignored local paths."
        ),
    },
]

clinical_interpretation_summary = pd.DataFrame(interpretation_rows)
display(clinical_interpretation_summary)


,evaluation_set,weighting,rows_or_groups,mean_foreground_dice,disc_dice,cup_dice,cdr_mae,mask_source
0,public_held_out_test,image_weighted,722,0.817976,0.839950,0.796002,0.063602,public_ground_truth
1,clinical_psd_derived,image_weighted,59,0.317713,0.377686,0.257740,0.290072,red_blue_overlay_extraction
2,clinical_psd_derived,patient_weighted,20,0.242017,0.267402,0.216632,0.283353,red_blue_overlay_extraction


,item,interpretation
0,selected_public_model,final_unetplusplus_resnet18_small_affine_virtu...
1,clinical_quantitative_subset,59 mask-ready PSD-derived clinical samples acr...
2,metric_caveat,Clinical masks are approximate labels extracte...
3,privacy_caveat,Committed outputs are aggregate-only. Raw clin...


## 08.15 — Final privacy and output audit

Confirm that committed outputs contain aggregate summaries only and that private clinical artifacts remain ignored.


In [15]:
# 08.15 — Final privacy and output audit
import subprocess

committed_outputs = [
    CLINICAL_EXTRACTION_SUMMARY_PATH,
    CLINICAL_DATASET_SUMMARY_PATH,
    CLINICAL_REBUILD_HISTORY_PATH,
    CLINICAL_REBUILD_METADATA_PATH,
    CLINICAL_EVALUATION_SUMMARY_PATH,
    CLINICAL_PATIENT_WEIGHTED_SUMMARY_PATH,
]

private_outputs = [
    PRIVATE_CLINICAL_MANIFEST_PATH,
    PRIVATE_CLINICAL_EXTRACTION_SUMMARY_PATH,
    PRIVATE_QA_CONTACT_SHEET_PATH,
    PRIVATE_CLINICAL_IMAGE_LEVEL_METRICS_PATH,
    PRIVATE_MODEL_REBUILD_CHECKPOINT_PATH,
    PRIVATE_CLEAN_DIR,
    PRIVATE_MASK_DIR,
]

committed_audit = pd.DataFrame(
    [
        {
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else np.nan,
            "expected_committed": True,
        }
        for path in committed_outputs
    ]
)

private_audit_rows = []
for path in private_outputs:
    result = subprocess.run(
        ["git", "check-ignore", "-q", str(path)],
        cwd=PROJECT_ROOT,
        check=False,
    )
    private_audit_rows.append(
        {
            "path": str(path.relative_to(PROJECT_ROOT)),
            "exists": path.exists(),
            "git_ignored": result.returncode == 0,
            "expected_committed": False,
        }
    )

private_audit = pd.DataFrame(private_audit_rows)

display(committed_audit)
display(private_audit)

if not committed_audit["exists"].all():
    missing = committed_audit.loc[~committed_audit["exists"], "path"].tolist()
    raise FileNotFoundError(f"Missing expected committed outputs: {missing}")

if not private_audit["git_ignored"].all():
    not_ignored = private_audit.loc[~private_audit["git_ignored"], "path"].tolist()
    raise RuntimeError(f"Private outputs are not ignored: {not_ignored}")

print("Notebook 08 clinical generalization complete.")


,path,exists,size_bytes,expected_committed
0,reports/data_audit/clinical_psd_annotation_ext...,True,4749,True
1,reports/data_audit/clinical_generalization_dat...,True,458,True
2,reports/training/clinical_generalization_model...,True,2310,True
3,reports/training/clinical_generalization_model...,True,693,True
4,reports/training/clinical_generalization_evalu...,True,1626,True
5,reports/training/clinical_generalization_patie...,True,1666,True


,path,exists,git_ignored,expected_committed
0,data/interim/private_clinical_generalization/c...,True,True,False
1,data/interim/private_clinical_generalization/c...,True,True,False
2,data/interim/private_clinical_generalization/q...,True,True,False
3,data/interim/private_clinical_generalization/c...,True,True,False
4,data/interim/private_clinical_generalization/s...,True,True,False
5,data/interim/private_clinical_generalization/c...,True,True,False
6,data/interim/private_clinical_generalization/d...,True,True,False


Notebook 08 clinical generalization complete.
